In [1]:
%pip install llama-index-llms-together

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install llama-index

In [15]:
import os
from llama_index.llms.together import TogetherLLM
from llama_index.core.llms import ChatMessage

os.environ["TOGETHER_API_KEY"] = '6379c9bdda7f5f5cbb14abbce2046110e50b0cc7aee25eb809da8bc6fa2a43d0'

/home/thanhtruongtran/Desktop/RecSys/post-generation/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
cd ..


/home/thanhtruongtran/Desktop/RecSys/post-generation


/home/thanhtruongtran/Desktop/RecSys/post-generation/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [19]:
from models.embedding_models import EmbeddingModel
from llama_index.core.vector_stores import VectorStoreQuery
from databases.postgresql import PostgresDB
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# initialize
embed_model = EmbeddingModel()._embed_model
vector_store=PostgresDB().vector_store

#test
query_str = "aave, gpt"
query_embedding = embed_model.get_query_embedding(query_str)
query_mode = "default"
vector_store_query = VectorStoreQuery(
    query_embedding=query_embedding, similarity_top_k=20, mode=query_mode
)

query_result = vector_store.query(vector_store_query)
tweet_simi =dict()
for i in range(0, 20):
    tweet_simi[f'post {i}'] = query_result.nodes[i].get_content()

/home/thanhtruongtran/Desktop/RecSys/post-generation/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/thanhtruongtran/Desktop/RecSys/post-generation/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [20]:
import random
llm = TogetherLLM(
    model='mistralai/mixtral-8x7b-instruct-v0.1', api_key='6379c9bdda7f5f5cbb14abbce2046110e50b0cc7aee25eb809da8bc6fa2a43d0',
    temperature=0.75, max_length=500, top_p= 1
)

prompt = [
    ChatMessage(
        role="user", content="""
            You are a professional marketer and your task is to create a post for a social media platform.
            Now, I will give you my requirement and after I say 'Start!!!', you will start your task.
        """
    ),
    ChatMessage(
        role="system", content="Yeah, I can help you with that. What is your requirement?"
    ),
    ChatMessage(
        role="user", content="""
            I want to create a post for a social media platform with the keywords: {query_str};
            You must write a post suitable for a social media platform.
            - If I want to create a post for Twitter, please limit your post to maximum 280 characters.
            - If I want to create a post for Telegram or Discord, consider your post with longer content and more detailed information.
            I will provide you with some examples of posts. Please wait a moment.
        """
    ),
    ChatMessage(
        role="system", content="Sure, I can help you with that. Please provide me with the social media platform you want to create a post for, and examples of posts."
    ),
    ChatMessage(
        role="user", content=f"""
            I want to create a post for Twitter. 
            Here are some examples of posts:
                - Post1 content: {tweet_simi[f'post {0}']}
                - Post2 content: {tweet_simi[f'post {1}']}
                - Post3 content: {tweet_simi[f'post {2}']}
                - Post4 content: {tweet_simi[f'post {3}']}
                - Post5 content: {tweet_simi[f'post {4}']}
            Please write a post in the same style and layout as the above posts. Your post should only contain content without any unnecessary information such as "Author", "Note", "Introduction", or "Conclusion".
        """
    ),
    ChatMessage(
        role="system", content="Great! I have understood your requirement. Do you have any other requirements?"
    ),
    ChatMessage(
        role="user", content="""
            Remember that:
                - Your post should only contain content, no additional information or introduction or conclusion.
                - Limit the number of "\\n" in your post to a maximum of 2. Use line breaks only when you want to write a new part of your post.
        """
    ),
    ChatMessage(
        role="system", content="Okay, I have understood your requirement. Let's start your task now."
    ),
    ChatMessage(
        role="user", content="Start!!!"
    ),
]

result = llm.stream_chat(prompt)


In [21]:
for r in result:
    print(r.delta, end="")

 Join us today, 10am EDT, to learn about @aave's upcoming features with our experts:

👉 @0xDeFiDevin from @GammaSwapLabs
👉 @CupOJoseph from @open_dsecurities
👉 @useteller from @useteller

👉Set your reminder here 👇
t.co/5OpODy2aFZ https://t.co/rXp7SDcpAp

<end of post>

In [4]:
print("🤖 #TRAVAGPT V2.0 IS OFFICIALLY LIVE FOR EVERYONE🎉🎉🎉\n\nThe limit has been lifted — anyone who has at least $5 of $TRAVA in their Wallet can now experience TravaGPT 🎇\n\n💬 Try it out ➡️ https://t.co/0MDKXMGRU9\n\n🔥 Help TravaGPT improve by filling this ➡️ https://t.co/w1SHg4QnLP https://t.co/2NuJZoBTxv")

🤖 #TRAVAGPT V2.0 IS OFFICIALLY LIVE FOR EVERYONE🎉🎉🎉

The limit has been lifted — anyone who has at least $5 of $TRAVA in their Wallet can now experience TravaGPT 🎇

💬 Try it out ➡️ https://t.co/0MDKXMGRU9

🔥 Help TravaGPT improve by filling this ➡️ https://t.co/w1SHg4QnLP https://t.co/2NuJZoBTxv


In [24]:
import re

def clean_generated_content(content):
    # Remove system notes or character count indicators enclosed in parentheses or brackets
    cleaned_content = re.sub(r'<<.*?>>', '', content)
    cleaned_content = re.sub(r'\([^)]*\)', '', cleaned_content)
    cleaned_content = re.sub(r'\[[^\]]*\]', '', cleaned_content)
    
    # Normalize whitespace, keeping double newlines
    cleaned_content = re.sub(r'[ \t]+', ' ', cleaned_content)
    cleaned_content = re.sub(r' *\n\n *', '\n\n', cleaned_content)
    cleaned_content = cleaned_content.strip()
    
    return cleaned_content

def split_content_into_parts(content, num_parts=3):
    # Split the content into parts by double newlines
    parts = content.split('\n\n')
    
    # If there are fewer parts than expected, fill the remaining parts with empty strings
    if len(parts) < num_parts:
        parts.extend([''] * (num_parts - len(parts)))
    
    # Combine the parts into the desired format with single newline separating them
    combined_content = '\n\n'.join(parts[:num_parts])
    
    return combined_content

# Example generated posts
posts = [
    "<<SYS>>\nExciting news for Trava Finance community! 🚀 The $TRAVA token is now available for trading on GPT! 💱\n\n📈 The $TRAVA markets are showing strong growth and availability on various platforms and products.\n🔒 Secure your $TRAVA tokens now and be a part of the future of decentralized finance!\n\n👉 Check out the link below for more information:\n<https://t.co/your-link-here>\n\n#TravaFinance #GPT #DeFi #CryptoTrading #TRAVA\n<<SYS>>\n\nNote: Please replace \"<https://t.co/your-link-here>\" with the actual link to the trading platform."
]

# Process and split each post into parts
for post in posts:
    cleaned_post = clean_generated_content(post)
    # split_post = split_content_into_parts(cleaned_post)
    print(cleaned_post)
    print("\n---------------------------------------\n")


Exciting news for Trava Finance community! 🚀 The $TRAVA token is now available for trading on GPT! 💱

📈 The $TRAVA markets are showing strong growth and availability on various platforms and products.
🔒 Secure your $TRAVA tokens now and be a part of the future of decentralized finance!

👉 Check out the link below for more information:
<https://t.co/your-link-here>

#TravaFinance #GPT #DeFi #CryptoTrading #TRAVA


Note: Please replace "<https://t.co/your-link-here>" with the actual link to the trading platform.

---------------------------------------



In [6]:
print(len("Introducing DeFi AI Fusion ⚛️, the revolutionary convergence of decentralized finance and artificial intelligence.\n\nImagine a world where AI-driven insights empower DeFi traders to make data-backed decisions, maximizing returns and minimizing risks.\n\nDeFi AI Fusion is the future of decentralized finance, where machine learning algorithms and natural language processing converge to create a new era of intelligent trading.\n\nLearn more about DeFi AI Fusion and its potential to transform the DeFi landscape: https://blog.defiai.io/defi-ai-fusion-the-future-of-decentralized-finance/\n\nJoin the DeFi AI Fusion community to stay updated on the latest developments and insights: https://t.me/defiaifusion"))

701


In [8]:
print(len(""" Keep yourself in the loop with the freshest updates from FlowX Weekly News! 📰 (12-19/5)

Here's what's making waves:
✅ Discover the Top 5 Tokens, Trading Pairs on trading volumes in the past 24 hours.
✅ FlowX Finance boasts a Total Value Locked (TVL) of $31.27M.
✅ Get ready for the grand reveal of FlowX V3 - Concentrated Liquidity.
✅ FlowX CLMM successfully passed Movebit audit

🔗 Spread the word: https://x.com/FlowX_finance/status/1792514146002718792

Join us on this exhilarating journey and let's conquer new milestones together! 🔥

"""))

541


In [10]:
print("🚀 Unlock the Power of DeFi with AI 🔹\n\nDear DeFi enthusiasts,\n\nExplore the intersection of Decentralized Finance and Artificial Intelligence, and discover how AI is revolutionizing the DeFi landscape. From predictive analytics to automated trading, AI is unlocking new opportunities for DeFi investors and developers.\n\n💡 Dive into the world of DeFi and AI, and learn how to harness the power of machine learning to drive growth and innovation in the DeFi space.\n\n📺 Watch our latest video to learn more: https://youtu.be/yourvideoURL\n\n🤟 1inch | Twitter | Discord | Medium | Reddit | YouTube\n\nDownload the 1inch Wallet: 🛒 iOS | 🛒 Android")

🚀 Unlock the Power of DeFi with AI 🔹

Dear DeFi enthusiasts,

Explore the intersection of Decentralized Finance and Artificial Intelligence, and discover how AI is revolutionizing the DeFi landscape. From predictive analytics to automated trading, AI is unlocking new opportunities for DeFi investors and developers.

💡 Dive into the world of DeFi and AI, and learn how to harness the power of machine learning to drive growth and innovation in the DeFi space.

📺 Watch our latest video to learn more: https://youtu.be/yourvideoURL

🤟 1inch | Twitter | Discord | Medium | Reddit | YouTube

Download the 1inch Wallet: 🛒 iOS | 🛒 Android


In [11]:
print("🚀 Revolutionizing DeFi with AI: Unlocking New Possibilities 🚀\n\nAt the forefront of DeFi innovation, we're excited to announce a strategic partnership between SingularityNET and Minswap Labs, blending AI with DeFi to push boundaries! 🤝\n\n➜ Developing integrations that utilize SingularityNET's AI to enhance the Minswap platform.\n➜ Leveraging Minswap DEX's infrastructure to improve the accessibility and usability of SingularityNET's native token.\n\nDive deeper into how we're merging AI with DeFi to create a new era of decentralized finance! 👇\nhttps://blog.singularitynet.io/9780f27a459d\n\n👉 Learn more about our vision for AI-powered DeFi and stay updated on our progress! \nWebsite | Telegram | Twitter | Medium | Reddit | Discord | YouTube | Forum | Zealy")

🚀 Revolutionizing DeFi with AI: Unlocking New Possibilities 🚀

At the forefront of DeFi innovation, we're excited to announce a strategic partnership between SingularityNET and Minswap Labs, blending AI with DeFi to push boundaries! 🤝

➜ Developing integrations that utilize SingularityNET's AI to enhance the Minswap platform.
➜ Leveraging Minswap DEX's infrastructure to improve the accessibility and usability of SingularityNET's native token.

Dive deeper into how we're merging AI with DeFi to create a new era of decentralized finance! 👇
https://blog.singularitynet.io/9780f27a459d

👉 Learn more about our vision for AI-powered DeFi and stay updated on our progress! 
Website | Telegram | Twitter | Medium | Reddit | Discord | YouTube | Forum | Zealy


In [12]:
print("📢 PEOPLE ARE TALKING ABOUT THE AFTERMATH: WHAT YOU NEED TO KNOW 🏛\n\nDear community,\n\nIn this regard, we would like to inform you that our contract update is scheduled to take place this Friday , in line with our Week 2 incentive timeline. The contract will be redeployed, and all pairs currently in Farms will be moved to Finished section and subsequently terminated.\n\nAdditionally, a small number of accounts that have not yet removed their LPs currently marked as Finished have been identified. After the Week 2 incentive timeline, these LPs will not be visible through the UI, and a separate withdrawal process will be required. We kindly ask the affected parties to contact us via our support email at support@dgswap.io.\n\nWe will provide further details on the exact timing and scale of the Week 2 incentives in a subsequent announcement.\n\nThank you for your understanding! 🙏\nWebsite / Docs / Medium / Twitter")

📢 PEOPLE ARE TALKING ABOUT THE AFTERMATH: WHAT YOU NEED TO KNOW 🏛

Dear community,

In this regard, we would like to inform you that our contract update is scheduled to take place this Friday , in line with our Week 2 incentive timeline. The contract will be redeployed, and all pairs currently in Farms will be moved to Finished section and subsequently terminated.

Additionally, a small number of accounts that have not yet removed their LPs currently marked as Finished have been identified. After the Week 2 incentive timeline, these LPs will not be visible through the UI, and a separate withdrawal process will be required. We kindly ask the affected parties to contact us via our support email at support@dgswap.io.

We will provide further details on the exact timing and scale of the Week 2 incentives in a subsequent announcement.

Thank you for your understanding! 🙏
Website / Docs / Medium / Twitter


In [1]:
print(len(""""🔥 Explore the Future of DeFi with 1inch! 🚀\n\nAt the 1inch Davos meetup, our panel discussed \"Institutional Adoption of DeFi 2024: What We Need Going Forward.\" 🌐\n\nModerated by Christine Lee, the insightful panel featured Romain Braud, Néstor Palao, and Svetlana Sailer, who delved into institutional DeFi adoption, tackling regulation, scalability, and product evolution. 🙌\n\nImagine a DeFi world powered by smart contracts and operating 24/7 globally. ✨\n\nRead more: https://blog.1inch.io/the-1inch-davos-meetup-institutional-adoption-of-decentralized-finance-2024/\n\n1inch | Twitter | Discord | Medium | Reddit | YouTube\n\nDownload the 1inch Wallet: 🛒 iOS | 🛒 Android"
          """))

676


In [4]:
print("📢 AFTERMATH OF THE RISE OF STARS 🚀\n\nDear Admirals,\n\nRise of Stars, which kicked off on February 25th, 2022 through WEMIX PLAY, has reached its conclusion to prepare the farewell to admirals on February 29th, 2024.\n\nEven after the service termination, we will continue to provide support for the Silthereum token.\n\nPlease refer to the contents below for the Rise of Stars service termination details.\n\n\n• Termination date : February 29th, 2024\n• Paid item purchase closure date : January 30th, 2024\n• In-game blockchain discontinuation date : February 20th, 2024\n• Silthereum Staking Season 2 end date : March 8th, 2024\n\n \n• Token, DEX, Swap, and Pool services in the PLAY Wallet will continue to operate as usual.\n\nIf you have any questions related to the game, refund, or any other matter, including the provided information, please send an inquiry to the customer support listed below. \n\n\n• Rise of Stars Customer Support : ros_cs@wemade.com \n• WEMIX PLAY Support Center : https://wemixplay.zendesk.com/ \n\nRise of Stars has come this far thanks to everyone's love and support.\nOnce again, we appreciate your affection and your role in the journey of Rise of Stars.\n\nThank you.")

📢 AFTERMATH OF THE RISE OF STARS 🚀

Dear Admirals,

Rise of Stars, which kicked off on February 25th, 2022 through WEMIX PLAY, has reached its conclusion to prepare the farewell to admirals on February 29th, 2024.

Even after the service termination, we will continue to provide support for the Silthereum token.

Please refer to the contents below for the Rise of Stars service termination details.


• Termination date : February 29th, 2024
• Paid item purchase closure date : January 30th, 2024
• In-game blockchain discontinuation date : February 20th, 2024
• Silthereum Staking Season 2 end date : March 8th, 2024

 
• Token, DEX, Swap, and Pool services in the PLAY Wallet will continue to operate as usual.

If you have any questions related to the game, refund, or any other matter, including the provided information, please send an inquiry to the customer support listed below. 


• Rise of Stars Customer Support : ros_cs@wemade.com 
• WEMIX PLAY Support Center : https://wemixplay.zendesk

In [16]:
print("⚔️ TRAMLINES JUST GOT BETTER 🚉\n\nTrava Knight Fellows,\n\nWe are excited to announce that our team has been working on enhancing the user experience in Trava Station, focusing on the Pancake Swap section.\n\n🥞 Tomorrow, more actions for Pancake Swap V2 will be added to Freight, prompting new Routes in Tramlines.\n\n📦 Check out Trava Station now and wait for the update IN ONE DAY ➡️ https://app.trava.finance/station?group=pancake\n\n———\n\n💬 Challenge TravaGPT V2.1 NOW!\n👥 Our Official Channels: Website | Telegram | Announcement | Twitter | Medium | Docs | Reddit | Discord | Youtube | Forum | Zealy | TaskOn")

⚔️ TRAMLINES JUST GOT BETTER 🚉

Trava Knight Fellows,

We are excited to announce that our team has been working on enhancing the user experience in Trava Station, focusing on the Pancake Swap section.

🥞 Tomorrow, more actions for Pancake Swap V2 will be added to Freight, prompting new Routes in Tramlines.

📦 Check out Trava Station now and wait for the update IN ONE DAY ➡️ https://app.trava.finance/station?group=pancake

———

💬 Challenge TravaGPT V2.1 NOW!
👥 Our Official Channels: Website | Telegram | Announcement | Twitter | Medium | Docs | Reddit | Discord | Youtube | Forum | Zealy | TaskOn
